# Aula 6 — Transfer Learning com MobileNetV2

Execute **Runtime > Run all** no Google Colab com GPU selecionada. O notebook baixa o repositório e usa o dataset de classificação já organizado em `train/` e `validation/`. Ao final, deixe visíveis a estrutura de pastas, as 10 épocas e `model.evaluate(val_ds)` para as capturas solicitadas.

In [ ]:
from pathlib import Path
import os
import subprocess

REPOSITORY_URL = 'https://github.com/gvenancio12/yolo-edge-api.git'
WORKDIR = Path('/content/yolo-edge-api-aula6')
if not WORKDIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(WORKDIR)], check=True)
else:
    subprocess.run(['git', '-C', str(WORKDIR), 'pull', '--ff-only'], check=True)
os.chdir(WORKDIR)
print('Diretório de trabalho:', Path.cwd())

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPUs disponíveis:', gpus)
if not gpus:
    raise RuntimeError('GPU não encontrada. No Colab: Runtime > Change runtime type > T4 GPU, depois reinicie e execute tudo.')

In [ ]:
from collections import Counter

DATASET_ROOT = Path('datasets/aula6-classification')
TRAIN_DIR = DATASET_ROOT / 'train'
VALIDATION_DIR = DATASET_ROOT / 'validation'
assert TRAIN_DIR.is_dir() and VALIDATION_DIR.is_dir(), 'Dataset de classificação ausente no clone.'

for split_dir in (TRAIN_DIR, VALIDATION_DIR):
    counts = {class_dir.name: len(list(class_dir.glob('*'))) for class_dir in sorted(split_dir.iterdir()) if class_dir.is_dir()}
    print(f'{split_dir}/')
    for class_name, amount in counts.items():
        print(f'  {class_name}/: {amount} imagens')
    assert len(counts) >= 2 and all(counts.values()), 'São necessárias pelo menos duas classes com imagens.'

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    VALIDATION_DIR, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=False
)
class_names = train_ds.class_names
assert class_names == val_ds.class_names
print('Classes:', class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,), include_top=False, weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs, name='mobilenetv2_epi_transfer')
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)
model.summary()

In [ ]:
EPOCHS = 10
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)
final_accuracy = history.history['accuracy'][-1]
final_val_accuracy = history.history['val_accuracy'][-1]
print(f'Acurácia final após {EPOCHS} épocas — treino: {final_accuracy:.4f}; validação: {final_val_accuracy:.4f}')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(history.history['accuracy'], label='treino')
plt.plot(history.history['val_accuracy'], label='validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.title('MobileNetV2 — Transfer Learning')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
evaluation = model.evaluate(val_ds, return_dict=True)
print('Resultado de model.evaluate(val_ds):', evaluation)